# 03 — Build User Level Dataset

Build the table the entire project is about.

Goal:
Create a single analytical table containing user attributes, advertising exposure, and conversion outcomes for incrementality analysis.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

sns.set_theme(style='whitegrid')
%matplotlib inline

In [22]:
# Load the 3 tables
dim_users = pd.read_csv('../data/raw/dim_users.csv')

fact_ad_exposures = pd.read_csv('../data/processed/fact_ad_exposures.csv')

fact_conversions = pd.read_csv('../data/processed/fact_conversions.csv') 

print(dim_users.shape)
print(fact_ad_exposures.shape)
print(fact_conversions.shape)   

(25000, 16)
(9718, 16)
(3650, 6)


In [23]:
# Join Exposure data
user_level_dataset = (
    dim_users.merge(
        fact_ad_exposures,
        on = 'user_id',
        how = 'left'
    )
)


In [24]:
# Join Conversion data
user_level_dataset = (
    user_level_dataset.merge(
        fact_conversions,
        on = 'user_id',
        how = 'left'
    )
)

In [25]:
print(user_level_dataset.shape)

(25000, 36)


In [26]:
# Validate Population Preservation
print(
    'Rows:',
    len(user_level_dataset)
)

print(
    'Unique Users:',
    user_level_dataset['user_id'].nunique()
)

print(
    'Duplicate Users:',
    user_level_dataset['user_id'].duplicated().sum()
)

Rows: 25000
Unique Users: 25000
Duplicate Users: 0


In [27]:
# Fill Exposure Fields
exposure_cols = [
    'exposed_flag',
    'total_ad_events',
    'total_impressions',
    'total_clicks',
    'total_video_starts',
    'total_video_completes',
    'days_exposed',
    'ctv_impressions',
    'meta_impressions',
    'youtube_impressions',
    'tiktok_impressions',
    'programmatic_display_impressions'
]

user_level_dataset[exposure_cols] = (
    user_level_dataset[exposure_cols]
    .fillna(0)
)

In [28]:
# Fill Conversion Fields
user_level_dataset['converted_flag'] = (
    user_level_dataset['converted_flag']
    .fillna(0)
)

In [29]:
# Sanity Check: Exposure
print(
    user_level_dataset['exposed_flag'].value_counts()
)

exposed_flag
0.00    15282
1.00     9718
Name: count, dtype: int64


In [30]:
# Sanity Check: Conversion
print(
    user_level_dataset['converted_flag'].value_counts()
)

converted_flag
0.00    21350
1.00     3650
Name: count, dtype: int64


In [31]:
print(dim_users["user_id"].head())
print(fact_conversions["user_id"].head())

0    U1000000
1    U1000001
2    U1000002
3    U1000003
4    U1000004
Name: user_id, dtype: str
0    U1000004
1    U1000014
2    U1000023
3    U1000041
4    U1000073
Name: user_id, dtype: str


In [32]:
print(
    len(
        set(dim_users["user_id"])
        &
        set(fact_conversions["user_id"])
    )
)

3650


In [ ]:
# Build user_level_dataset

# join exposure data
user_level_dataset = dim_users.merge(
    fact_ad_exposures,
    on='user_id',
    how='left'
)

print(user_level_dataset.shape)

print(
    user_level_dataset["user_id"]
    .nunique()
)

(25000, 31)
25000


In [36]:
# join conversion data
user_level_dataset = user_level_dataset.merge(
    fact_conversions,
    on='user_id',
    how='left'
)

print(user_level_dataset.shape)

print(
    user_level_dataset["user_id"]
    .nunique()
)

print(
    user_level_dataset["user_id"]
    .duplicated()
    .sum()
)

(25000, 36)
25000
0


In [37]:
user_level_dataset.head()

,user_id,account_created_date,age_band,gender,dma,state,region,income_band,device_preference,streaming_household_flag,prior_subscriber_flag,lapsed_subscriber_flag,prior_trial_flag,genre_affinity,customer_segment,historical_engagement_score,total_ad_events,total_impressions,total_clicks,total_video_starts,total_video_completes,first_exposure,last_exposure_date,days_exposed,exposed_flag,ctv_impressions,meta_impressions,programmatic_display_impressions,tiktok_impressions,youtube_impressions,frequency_bucket,converted_flag,first_conversion_date,plan_type,conversion_amount,subscription_status
0,U1000000,2022-08-20,45-54,F,NaN,NaN,NaN,30-60k,Mobile,1,1,0,1,Comedy,Premium Loyalist,24.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U1000001,2023-12-11,45-54,M,Spokane,WA,West,60-100k,Connected TV,0,0,0,0,Kids & Family,Casual Streamer,25.60,9.00,9.00,2.00,6.00,3.00,2024-04-02 04:33:40,2024-04-27 22:06:08,26.00,1.00,1.00,4.00,1.00,1.00,2.00,1-10,NaN,NaN,NaN,NaN,NaN
2,U1000002,2023-02-17,35-44,M,Fresno,CA,West,NaN,Mobile,1,0,0,0,Sci-Fi,New Prospect,19.70,66.00,66.00,0.00,54.00,26.00,2024-04-01 20:59:38,2024-04-28 19:51:52,27.00,1.00,18.00,16.00,4.00,17.00,11.00,61-80,NaN,NaN,NaN,NaN,NaN
3,U1000003,2022-09-26,35-44,M,Charlotte,NC,South,30-60k,Mobile,0,1,1,1,Sci-Fi,Lapsed Subscriber,40.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,U1000004,2024-01-15,35-44,NaN,Miami-Ft. Lauderdale,FL,South,30-60k,Desktop,0,0,0,0,Comedy,New Prospect,14.60,19.00,19.00,0.00,13.00,8.00,2024-04-01 18:14:56,2024-04-28 20:34:23,28.00,1.00,4.00,5.00,2.00,5.00,3.00,11-20,1.00,2023-01-01,quarterly,33.63,canceled


In [38]:
user_level_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 36 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   user_id                           25000 non-null  str    
 1   account_created_date              25000 non-null  str    
 2   age_band                          24727 non-null  str    
 3   gender                            23948 non-null  str    
 4   dma                               24518 non-null  str    
 5   state                             24518 non-null  str    
 6   region                            24518 non-null  str    
 7   income_band                       22301 non-null  str    
 8   device_preference                 25000 non-null  str    
 9   streaming_household_flag          25000 non-null  int64  
 10  prior_subscriber_flag             25000 non-null  int64  
 11  lapsed_subscriber_flag            25000 non-null  int64  
 12  prior_trial_fla